In [5]:
import pandas as pd
import h5py
import numpy as np
from pathlib import Path

BASE = Path("/Volumes/Extreme SSD/stream_stead/data_stead")
CSV_PATH = BASE / "merge.csv"
H5_PATH  = BASE / "merge.hdf5"

# 1) Load metadata STEAD
df = pd.read_csv(CSV_PATH)
print("Total rows in STEAD:", len(df))


/var/folders/lt/2mkl6ry53ll9fdk2br6skfgw0000gn/T/ipykernel_41068/1736567029.py:11: DtypeWarning: Columns (7,11,13,14,24,25,26,30,31) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_PATH)


Total rows in STEAD: 1265657


In [8]:
# 2) Filter event near-field
df_ev = df[
    (df.trace_category == "earthquake_local") &
    (df.source_distance_km <= 50) &
    (df.source_magnitude >= 5)
].copy()

# 3) Filter noise (multi-station, random)
df_ns = df[df.trace_category == "noise"].copy()

# (opsional) sampling agar tidak kebanyakan
N_EV = 20000
N_NS = 20000

df_ev = df_ev.sample(min(N_EV, len(df_ev)), random_state=42)
df_ns = df_ns.sample(min(N_NS, len(df_ns)), random_state=42)

print("Event selected:", len(df_ev))
print("Noise selected:", len(df_ns))


Event selected: 143
Noise selected: 20000


In [11]:
import scipy.signal as sig
from scipy.signal.windows import tukey

def load_waveform(h5_file, trace_name):
    ds = h5_file.get("data/" + str(trace_name))
    if ds is None:
        return None
    data = np.array(ds)  # shape (6000, 3)
    if data.shape != (6000, 3):
        return None
    return data

def preprocess_trace(x, bandpass=True):
    x = x.astype(np.float32)

    # detrend
    x = sig.detrend(x, axis=0, type="linear")

    # taper (versi stabil)
    win = tukey(x.shape[0], alpha=0.05)
    x = x * win[:, None]

    # bandpass 1–20 Hz
    if bandpass:
        fs = 100.0
        sos = sig.butter(4, [1.0, 20.0], btype="bandpass", fs=fs, output="sos")
        x = sig.sosfiltfilt(sos, x, axis=0)

    # normalisasi robust
    std = np.std(x)
    if std > 0:
        x = x / (std + 1e-6)

    # clipping
    x = np.clip(x, -5.0, 5.0)

    return x

def build_Xy(df_subset, label, h5_path):
    """
    df_subset: dataframe dengan kolom trace_name
    label: 0 = noise, 1 = event
    """
    X_list = []
    y_list = []

    with h5py.File(h5_path, "r") as f:
        for i, tr_name in enumerate(df_subset["trace_name"].values):
            data = load_waveform(f, tr_name)
            if data is None:
                continue
            x_proc = preprocess_trace(data)
            X_list.append(x_proc)
            y_list.append(label)

            if (i + 1) % 1000 == 0:
                print(f"{i+1} traces processed...")

    X = np.stack(X_list, axis=0)  # (N, 6000, 3)
    y = np.array(y_list, dtype=np.int64)
    return X, y

# 4) Bangun X_event, y_event
X_ev, y_ev = build_Xy(df_ev, label=1, h5_path=H5_PATH)
print("Event shape:", X_ev.shape, y_ev.shape)

# 5) Bangun X_noise, y_noise
X_ns, y_ns = build_Xy(df_ns, label=0, h5_path=H5_PATH)
print("Noise shape:", X_ns.shape, y_ns.shape)

# 6) Gabungkan & simpan ke SSD
X_all = np.concatenate([X_ev, X_ns], axis=0)
y_all = np.concatenate([y_ev, y_ns], axis=0)

np.save(BASE / "X_stead_ac3.npy", X_all)
np.save(BASE / "y_stead_ac3.npy", y_all)

print("Saved:", BASE / "X_stead_ac3.npy", BASE / "y_stead_ac3.npy")


Event shape: (143, 6000, 3) (143,)
1000 traces processed...
2000 traces processed...
3000 traces processed...
4000 traces processed...
5000 traces processed...
6000 traces processed...
7000 traces processed...
8000 traces processed...
9000 traces processed...
10000 traces processed...
11000 traces processed...
12000 traces processed...
13000 traces processed...
14000 traces processed...
15000 traces processed...
16000 traces processed...
17000 traces processed...
18000 traces processed...
19000 traces processed...
20000 traces processed...
Noise shape: (20000, 6000, 3) (20000,)
Saved: /Volumes/Extreme SSD/stream_stead/data_stead/X_stead_ac3.npy /Volumes/Extreme SSD/stream_stead/data_stead/y_stead_ac3.npy


In [12]:
import numpy as np
from pathlib import Path

BASE = Path("/Volumes/Extreme SSD/stream_stead/data_stead")

# 1) Load STEAD
X_stead = np.load(BASE / "X_stead_ac3.npy")
y_stead = np.load(BASE / "y_stead_ac3.npy")
d_stead = np.zeros_like(y_stead)  # domain id 0

print("STEAD:", X_stead.shape, y_stead.shape)

# 2) (Placeholder) Load K-NET
# Misal nanti kamu simpan di:
# /Volumes/Extreme SSD/knet_ac3/X_knet_ac3.npy
# /Volumes/Extreme SSD/knet_ac3/y_knet_ac3.npy

KNET_BASE = Path("/Volumes/Extreme SSD/knet_ac3")
try:
    X_knet = np.load(KNET_BASE / "X_knet_ac3.npy")
    y_knet = np.load(KNET_BASE / "y_knet_ac3.npy")
    d_knet = np.ones_like(y_knet)  # domain id 1
    print("K-NET:", X_knet.shape, y_knet.shape)
except FileNotFoundError:
    X_knet = None
    y_knet = None
    d_knet = None
    print("K-NET data not found, skipping for now.")

# 3) (Placeholder) Load Indonesia
ID_BASE = Path("/Volumes/Extreme SSD/id_ac3")
try:
    X_id = np.load(ID_BASE / "X_id_ac3.npy")
    y_id = np.load(ID_BASE / "y_id_ac3.npy")
    d_id = np.full_like(y_id, 2)  # domain id 2
    print("Indonesia:", X_id.shape, y_id.shape)
except FileNotFoundError:
    X_id = None
    y_id = None
    d_id = None
    print("Indonesia data not found, skipping for now.")

# 4) Gabungkan semua domain yang tersedia
X_list = [X_stead]
y_list = [y_stead]
d_list = [d_stead]

if X_knet is not None:
    X_list.append(X_knet)
    y_list.append(y_knet)
    d_list.append(d_knet)

if X_id is not None:
    X_list.append(X_id)
    y_list.append(y_id)
    d_list.append(d_id)

X_multi = np.concatenate(X_list, axis=0)
y_multi = np.concatenate(y_list, axis=0)
d_multi = np.concatenate(d_list, axis=0)

print("Multi-domain:", X_multi.shape, y_multi.shape, d_multi.shape)

# 5) Simpan
OUT_BASE = BASE  # atau folder lain
np.save(OUT_BASE / "X_multi_ac3.npy", X_multi)
np.save(OUT_BASE / "y_multi_ac3.npy", y_multi)
np.save(OUT_BASE / "d_multi_ac3.npy", d_multi)

print("Saved multi-domain dataset.")


STEAD: (20143, 6000, 3) (20143,)
K-NET data not found, skipping for now.
Indonesia data not found, skipping for now.
Multi-domain: (20143, 6000, 3) (20143,) (20143,)
Saved multi-domain dataset.


In [13]:
# ============================================================
# 📌 CELL — BUILDER MULTI-DOMAIN (STEAD + K-NET)
# Deskripsi:
# - Memuat dataset STEAD AC-3 dan K-NET AC-3
# - Membuat domain label (0=STEAD, 1=K-NET)
# - Menggabungkan seluruh data menjadi dataset multi-domain
# - Menyimpan X_multi_ac3.npy, y_multi_ac3.npy, d_multi_ac3.npy
# ============================================================

from tqdm import tqdm

# Load STEAD
print("Loading STEAD...")
X_stead = np.load(STEAD_BASE / "X_stead_ac3.npy")
y_stead = np.load(STEAD_BASE / "y_stead_ac3.npy")
d_stead = np.zeros_like(y_stead)

# Load K-NET
print("Loading K-NET...")
X_knet = np.load(KNET_OUT / "X_knet_ac3.npy")
y_knet = np.load(KNET_OUT / "y_knet_ac3.npy")
d_knet = np.ones_like(y_knet)

# Gabungkan multi-domain
print("Merging datasets...")
X_multi = np.concatenate([X_stead, X_knet], axis=0)
y_multi = np.concatenate([y_stead, y_knet], axis=0)
d_multi = np.concatenate([d_stead, d_knet], axis=0)

# Simpan
np.save(MULTI_OUT / "X_multi_ac3.npy", X_multi)
np.save(MULTI_OUT / "y_multi_ac3.npy", y_multi)
np.save(MULTI_OUT / "d_multi_ac3.npy", d_multi)

print("Saved multi-domain dataset:")
print("X:", X_multi.shape)
print("y:", y_multi.shape)
print("d:", d_multi.shape)


Loading STEAD...


NameError: name 'STEAD_BASE' is not defined